![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 15 -- Lab 2: Digits with a PyTorch Network

An Olympiad-style task: 1,797 scanned digits of 8 x 8 pixels, labels 0 to 9 for the training part, a hidden test part to predict. You first try the Week 2 way (columns typed by hand, then a line), then the raw pixels, then build and train a network in PyTorch, read its loss curve, look at the columns it invented, tune its one dial on validation rows, and write a submission file.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The scikit-learn digits (Alpaydin and Kaynak, 1998, UCI): `load_digits()` ships with scikit-learn, no download. The setup cell keeps 300 images aside as the hidden test set and splits the rest into training and validation rows.

---
# Setup

In [ ]:
# numpy, pandas, matplotlib, scikit-learn and torch are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q numpy pandas matplotlib scikit-learn torch

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
torch.manual_seed(0)

In [ ]:
digits = load_digits()
X_all, y_all = digits.data / 16.0, digits.target                 # pixels 0..16 -> 0..1
X_rest, X_test, y_rest, y_test = train_test_split(X_all, y_all, test_size=300, random_state=7, stratify=y_all)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.2, random_state=42, stratify=y_rest)
print(X_train.shape, X_val.shape, X_test.shape)

---
# Part 1 -- The Week 2 way

The *How We Did It Before* slide: a person chooses the columns, then one logistic regression per class learns its dials.

## Task 1: Look at the data

Show the first 10 training images in a row (`X_train[i].reshape(8, 8)` with `imshow(..., cmap="gray_r")`, the label as the title, axes off). Then print how many images each class has in `y_train` (`np.bincount`).

In [ ]:
# Your code here


## Task 2: Hand-made columns

Write `hand_features(A)` that turns an array of flattened images (`n` x 64) into 16 columns: the sum of each row and the sum of each column of the 8 x 8 image (`A.reshape(-1, 8, 8).sum(axis=2)` and `.sum(axis=1)`, joined with `np.c_`). Fit `LogisticRegression(max_iter=5000)` on these columns and print the validation accuracy. Expected about 0.87.

In [ ]:
# Your code here


## Task 3: The raw pixels

Fit `LogisticRegression(max_iter=5000)` on the 64 raw pixels of `X_train` and print the validation accuracy as `acc_raw`. Expected about 0.97. Which columns were better, yours or the pixels?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert acc_raw > acc_hand, 'raw pixels: LogisticRegression on X_train, scored on X_val'
print('Task 3 passed')

---
# Part 2 -- The network in PyTorch

The *PyTorch Basics* slides: tensors, `nn.Sequential`, the four-move loop.

## Task 4: Tensors

Turn the three feature arrays into `float32` tensors (`torch.tensor(X_train, dtype=torch.float32)`) named `Xt, Xv, Xte`, and the three label arrays into `int64` tensors `yt, yv, yte` (`torch.tensor(y_train)`). Print `Xt.shape, Xt.dtype, yt.dtype`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert Xt.dtype == torch.float32 and yt.dtype == torch.int64 and Xt.shape[1] == 64, 'tensors: features float32, labels int64'
print('Task 4 passed')

## Task 5: The model

Write `make_model(hidden)` returning `nn.Sequential(nn.Linear(64, hidden), nn.ReLU(), nn.Linear(hidden, 10))`. Create `model = make_model(32)`, print it, and print the number of parameters with `sum(p.numel() for p in model.parameters())`. Expected 2,410 (the *Same in Matrices* slide).

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_params == 2410, 'model: 64 * 32 + 32 + 32 * 10 + 10 dials'
print('Task 5 passed')

## Task 6: The untrained network

Create `loss_fn = nn.CrossEntropyLoss()`. Without training, compute the loss of `model(Xt)` against `yt` and the validation accuracy (`model(Xv).argmax(dim=1)` compared with `yv`, inside `torch.no_grad()`). The loss should be close to `ln(10) = 2.303` and the accuracy close to 0.1: why those two numbers?

In [ ]:
# Your code here


## Task 7: The training loop

Write `train(model, epochs, lr)` with the four moves: `opt = torch.optim.SGD(model.parameters(), lr=lr)`; each epoch: `logits = model(Xt)`, `loss = loss_fn(logits, yt)`, `opt.zero_grad()`, `loss.backward()`, `opt.step()`; after the step record `loss.item()` and the validation accuracy (under `torch.no_grad()`) in two lists, and return them. Run `train(model, 400, 0.5)` and print the final loss and validation accuracy. Expected: loss below 0.1, accuracy about 0.97.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert losses[-1] < 0.15 and accs[-1] > 0.93, 'train: SGD lr 0.5, zero_grad, backward, step, 400 epochs'
print('Task 7 passed')

## Task 8: Read the curves

Plot `losses` against the epoch on the left and `accs` on the right (two subplots, titles, axis labels). Draw `acc_raw` as a dashed horizontal line on the right plot. Is the loss still falling at epoch 400? Did the network pass the line?

In [ ]:
# Your code here


---
# Part 3 -- Inside, and the dial

The *What the Hidden Layer Learns* and *Limitations* slides.

## Task 9: The invented columns

Take the first layer's weights, `model[0].weight.detach().numpy()` (shape 32 x 64), and show the first 8 rows as 8 x 8 images (`cmap="RdBu_r"`, axes off). Each image is one hidden neuron's template.

In [ ]:
# Your code here


## Task 10: The dial: hidden size

For `hidden` in `[2, 8, 32, 128]`: build a fresh `make_model(hidden)` (call `torch.manual_seed(0)` first each time), train it for 400 epochs with lr 0.5, and store the final validation accuracy in a dict `results`. Print the dict. Which size is too small, and is the biggest the best?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert results[2] < results[32], 'hidden size: 2 neurons must score clearly below 32'
print('Task 10 passed')

---
# Part 4 -- Submit

An Olympiad hand-in: refit on everything you are allowed to use, predict the hidden images, check the file.

## Task 11: Refit and predict the test images

Fit a fresh `make_model(32)` on the training and validation rows together (`torch.cat([Xt, Xv])`, `torch.cat([yt, yv])`): write the loop again inline for 400 epochs with lr 0.5, then predict `Xte` with `argmax` under `torch.no_grad()` into `pred_test` (a NumPy array of 300 labels). Print the test accuracy against `y_test` and the confusion matrix (`confusion_matrix(y_test, pred_test)`). Which two digits are confused most?

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert pred_test.shape == (300,) and accuracy_score(y_test, pred_test) > 0.93, 'final model: train on Xt + Xv, predict Xte with argmax'
print('Task 11 passed')

## Task 12: Write the submission

Save `pd.DataFrame({"id": np.arange(300), "label": pred_test})` to `submission.csv` with `index=False`, read it back, and print its shape and the first 5 rows. The file must have exactly 300 rows and integer labels 0 to 9.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert check.shape == (300, 2) and set(check['label'].unique()) <= set(range(10)), 'submission.csv: 300 rows, columns id and label'
print('Task 12 passed')

---
## Task 13: Reflect

Your 16 hand-made columns lost to the raw pixels, and the network roughly tied the line on these 8 x 8 images while the deck's 28 x 28 network won by 6 points. What changes between the two image sizes that makes the network's invented columns matter? The untrained loss was ln(10). What would a loss stuck at ln(10) after 400 epochs tell you, and which two things would you check first (Day 08 and Day 09)? You picked the hidden size on validation rows and reported the test accuracy once. Why not pick it on the test rows?

*Your answers here*

---
## Conclusion

- **Columns**: the 16 you typed lost to the 64 raw pixels; the network then invented 32 columns of its own, which you drew as templates.
- **PyTorch**: tensors with the right dtypes, `nn.Sequential`, `CrossEntropyLoss`, and the four-move loop with `zero_grad`, `backward`, `step`.
- **Reading the run**: the untrained loss is ln(10), the loss curve falls, the validation accuracy climbs and ties the line on tiny images.
- **Olympiad habits**: the one dial was chosen on validation rows, the final model was refitted on everything allowed, and the submission file was checked before hand-in.

---

Made By **Sattam Altwaim**